## Training Pipeline

### Installations


In [ ]:
%%capture
!pip install unsloth==2025.1.6 comet_ml==3.48.1

### Prepare Environment

In [18]:
import os
from getpass import getpass

hf_token = getpass("Enter the Hugging face api token for authentication, press enter to skip : ")
enable_hf = bool(hf_token)
print("Is huggingface enabled? : ", enable_hf)

comet_api_key = getpass("Enter the api key for comet ml, press enter to skip: ")
enable_comet = bool(comet_api_key)
print("Is cometml enabled? : ", enable_comet)
comet_project_name = "second-brain"

if enable_hf:
    os.environ["HF_TOKEN"] = hf_token
if comet_api_key:
    os.environ["COMET_API_KEY"] = comet_api_key
    os.environ["COMET_PROJECT_NAME"] = comet_project_name

Is huggingface enabled? :  False
Is cometml enabled? :  False


### Global Variables

In [10]:
import torch


def get_gpu_info() -> str | None:
    """Gets GPU device name if available.

    Returns:
        str | None: Name of the GPU device if available, None if no GPU is found.
    """
    if not torch.cuda.is_available():
        return None

    gpu_name = torch.cuda.get_device_properties(0).name

    return gpu_name


active_gpu_name = get_gpu_info()

print("GPU type:")
print(active_gpu_name)

GPU type:
None


In [11]:
dataset_id =(
    input("Enter the dataset_id of the huggingface_dataset(summarization one): ") 
    or "pauliusztin/second_brain_course_summarization_task"
)

print(f"{dataset_id=}")

dataset_id='pauliusztin/second_brain_course_summarization_task'


Depending on your GPU type, we must pick different variables, as training in 4bit (QLoRA) takes substantially longer than training in 16bit (LoRA). Thus, if you have a T4 Nivia GPU, which is available in Google's Colab free tier, to avoid waiting an eternity for the fine-tuning to complete, we will train for fewer steps (on T4, we cannot train with LoRA without encountering issues while fine-tuning).

In [ ]:
if active_gpu_name and "T4" in active_gpu_name:
    max_evaluation_samples = 8
elif active_gpu_name and ("A100" in active_gpu_name or "L4" in active_gpu_name):
    max_evaluation_samples = 70
elif active_gpu_name:
    max_evaluation_samples = 8
else:
    raise ValueError("No Nvidia GPU found.")

print("--- Parameters ---")
print(f"{max_evaluation_samples=}")

### Loading LLM Using Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

base_model = "Meta-Llama-3.1-8B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = f"unsloth/{base_model}",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit=load_in_4bit,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing="unsloth",  # True or "unsloth" for very long context
    random_state=3407,
    use_rslora=False, # rank stablized 
    loftq_config=None
)

In [ ]:
from datasets import load_dataset

alpaca_promt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are a helpful assistant specialized in summarizing documents. Generate a concise TL;DR summary in markdown format having a maximum of 512 characters of the key findings from the provided documents, highlighting the most significant insights

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token # must add EOS token to tell llm when to stop 

def format_prompts(examples):
    inputs = examples["instruction"]
    outputs = examples["answer"]
    
    texts = []
    
    for input, output in zip(inputs, outputs):
        text = alpaca_promt.format(input, output) + EOS_TOKEN
        
        texts.append(text)
        
    return {
        "text" : texts
    }

In [ ]:
dataset = load_dataset(dataset_id)
dataset = dataset.map(
    format_prompts,
    batched=True,
)

### Train the Model 

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=True,  # Can make training 5x faster for short sequences as it pads new sample instead of padding tokens.
    args=TrainingArguments(
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        # num_train_epochs=1,  # Set this for 1 full training run, while commenting out 'max_steps'.
        max_steps=max_steps,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="comet_ml" if enable_comet else "none",
    ),
)

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime'] / 60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

### Inference


In [ ]:
from transformers import TextStreamer

FastLanguageModel.for_inference(model)

text_streamer = TextStreamer(tokenizer)

def generate_summary(
    instruction: str, streaming: bool = True, trim_input_message: bool = True,
):
    
    message = alpaca_promt.format(
            instruction,
            ""
        )
    
    inputs = tokenizer([message], return_tensor="pt").to("cuda")
    
    if streaming:
        return model.generate(**inputs, streamer=text_streamer, max_new_tokens=256, use_cache=True)
        
    else:
        output_tokens = model.generate(**inputs, max_new_tokens=250, use_cache=True,) # returns all tokens input + output
        response = tokenizer.batch_decode(output_tokens, skip_special_tokens=True)[0] 
        
        if trim_input_message:
            return response[len(message) :]
        else:
            return response


In [ ]:
generate_summary(dataset["validation"][0]["instructions"], streaming=True)

In [ ]:
generate_summary(dataset["validation"][0]["instruction"], streaming=False)

NameError: name 'dataset' is not defined

In [ ]:
from huggingface_hub import HfApi

# save merged model
model_name = f"{base_model}/second-brain"
print(f"Saving with model_name : {model_name}")
model.save_pretrained_merged(
    model_name,
    tokenizer,
    save_method="merged_16bit"
)

# Saving to Hugging face

if enable_hf:
    hf_api = HfApi()
    user_info = hf_api.whoami(token=hf_token)
    huggingface_user = user_info["name"]
    print(f"Current huggingface user: {huggingface_user}")
    
    model.push_to_hub_merged(
        f"{huggingface_user}/{model_name}",
        tokenizer=tokenizer,
        save_method="merged_16bit",
        token=hf_token,
        private=True,
    )
    